# Black–Scholes

For a call option:

$$
C(S,t,T,\sigma^2,K) = S\,N(d_1) - K e^{-r(T-t)} N(d_2).
$$

For a put option:

$$
P(S,t,T,\sigma^2,K) = K e^{-r(T-t)} N(-d_2) - S N(-d_1).
$$

where

$$
d_1 = \frac{\log(S/K) + \left(r + 0.5\sigma^2\right)(T-t)}{\sigma\sqrt{T-t}},
\qquad
d_2 = d_1 - \sigma\sqrt{T-t}.
$$

# Greeks

## Call and Put

- **Delta** ($\Delta$)

$$
\frac{\partial C}{\partial S} = N(d_1),
\qquad
\frac{\partial P}{\partial S} = N(d_1) - 1.
$$

- **Gamma** ($\Gamma$)

$$
\frac{\partial^2 C}{\partial S^2} = \frac{\partial^2 P}{\partial S^2}
= \frac{\psi(d_1)}{S\sigma\sqrt{T-t}},
$$

for standard normal PDF $\psi(\cdot)$.

- **Vega** ($\nu$)

$$
\frac{\partial C}{\partial \sigma} = \frac{\partial P}{\partial \sigma}
= S\psi(d_1)\sqrt{T-t}.
$$

- **Rho** ($\rho$)

$$
\frac{\partial C}{\partial r} = K e^{-r(T-t)}(T-t)N(d_2),
\qquad
\frac{\partial P}{\partial r} = -K e^{-r(T-t)}(T-t)N(-d_2).
$$

- **Theta** ($\Theta$)

$$
\frac{\partial C}{\partial t}
= -\frac{\sigma S\psi(d_1)}{2\sqrt{T-t}} - rK e^{-r(T-t)}N(d_2),
$$

$$
\frac{\partial P}{\partial t}
= -\frac{\sigma S\psi(d_1)}{2\sqrt{T-t}} + rK e^{-r(T-t)}N(-d_2).
$$

## Long Straddle

$$
V = C(S,t,T,\sigma^2,K) + P(S,t,T,\sigma^2,K).
$$

- Delta ($\Delta$): $2N(d_1) - 1$  
- Gamma ($\Gamma$): $\frac{2\psi(d_1)}{S\sigma\sqrt{T-t}}$  
- Vega ($\nu$): $2S\psi(d_1)\sqrt{T-t}$  
- Rho ($\rho$): $K e^{-r(T-t)}(T-t)\left(N(d_2) - N(-d_2)\right)$  
- Theta ($\Theta$): $-\frac{\sigma S\psi(d_1)}{\sqrt{T-t}} - rK e^{-r(T-t)}\left(N(d_2) - N(-d_2)\right)$

In [1]:
# imports
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm
import scipy.stats as sp
import pandas as pd


# set maturity in years (30 days)
T = 30 / 365

# set current time
t = 0

# set spot price
S = 100

# set strike price (ATM)
K = 100

# set short rate
r = 0.05

# set volatility
sigma = 0.15


# define function to compute BS call and put price
def black_scholes(S, K, r, sigma, T, t=0):
    """
    A function to compute the Black Scholes prices for call and put options.
    Args:
    -----
            S: float, spot price
            K: float, strike price
            r: float, interest rate (continuously compounded)
            sigma: float, volatility
            T: float, maturity time (in years)
            t: float, current time (in years)
    Returns:
    --------
            bs_call, bs_put: tuple, a pair of prices (call, put)
    """

    # compute time to maturity
    tau = T - t

    # validate inputs
    if tau <= 0:
        raise ValueError("Require T > t so that time-to-maturity (T-t) is positive.")
    if S <= 0 or K <= 0:
        raise ValueError("Require S > 0 and K > 0.")
    if sigma <= 0:
        raise ValueError("Require sigma > 0.")

    # compute d1
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * tau) / (sigma * np.sqrt(tau))

    # compute d2
    d2 = d1 - sigma * np.sqrt(tau)

    # compute call price
    bs_call = S * norm.cdf(d1) - K * np.exp(-r * tau) * norm.cdf(d2)

    # compute put price
    bs_put = K * np.exp(-r * tau) * norm.cdf(-d2) - S * norm.cdf(-d1)

    # ceturn call and put prices
    return bs_call, bs_put


# define function to compute standard call/put Greeks and long-straddle Greeks
def greeks_call_put(S, K, r, sigma, T, t=0):
    """
    A function to compute analytic Greeks for Black–Scholes call and put options,
    and the associated long straddle (call + put) Greeks.
    Args:
    -----
            S: float, spot price
            K: float, strike price
            r: float, interest rate (continuously compounded)
            sigma: float, volatility
            T: float, maturity time (in years)
            t: float, current time (in years)
    Returns:
    --------
            cp_greeks: dict, Greeks for call/put:
                - delta_call, delta_put
                - gamma
                - vega
                - rho_call, rho_put
                - theta_call, theta_put
            st_greeks: dict, Greeks for long straddle (call + put):
                - delta_ls, gamma_ls, vega_ls, rho_ls, theta_ls
    """

    # compute time to maturity
    tau = T - t

    # validate inputs
    if tau <= 0:
        raise ValueError("Require T > t so that time-to-maturity (T-t) is positive.")
    if S <= 0 or K <= 0:
        raise ValueError("Require S > 0 and K > 0.")
    if sigma <= 0:
        raise ValueError("Require sigma > 0.")

    # compute d1
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * tau) / (sigma * np.sqrt(tau))

    # compute d2
    d2 = d1 - sigma * np.sqrt(tau)

    # compute standard normal pdf at d1
    phi_d1 = norm.pdf(d1)

    # compute call delta
    delta_call = norm.cdf(d1)

    # compute put delta
    delta_put = norm.cdf(d1) - 1.0

    # compute gamma
    gamma = phi_d1 / (S * sigma * np.sqrt(tau))

    # compute vega
    vega = S * phi_d1 * np.sqrt(tau)

    # compute call rho
    rho_call = K * tau * np.exp(-r * tau) * norm.cdf(d2)

    # compute put rho
    rho_put = -K * tau * np.exp(-r * tau) * norm.cdf(-d2)

    # compute call theta
    theta_call = -(S * phi_d1 * sigma) / (2.0 * np.sqrt(tau)) - r * K * np.exp(-r * tau) * norm.cdf(d2)

    # compute put theta
    theta_put = -(S * phi_d1 * sigma) / (2.0 * np.sqrt(tau)) + r * K * np.exp(-r * tau) * norm.cdf(-d2)

    # assemble call/put Greeks dictionary
    cp_greeks = {
        "delta_call": delta_call,
        "delta_put": delta_put,
        "gamma": gamma,
        "vega": vega,
        "rho_call": rho_call,
        "rho_put": rho_put,
        "theta_call": theta_call,
        "theta_put": theta_put,
    }

    # assemble long straddle Greeks (call + put)
    st_greeks = {
        "delta_ls": delta_call + delta_put,
        "gamma_ls": gamma + gamma,
        "vega_ls": vega + vega,
        "rho_ls": rho_call + rho_put,
        "theta_ls": theta_call + theta_put,
    }

    # return both dictionaries
    return cp_greeks, st_greeks



# compute call/put and straddle greeks
cp_greeks, st_greeks = greeks_call_put(S=S, K=K, r=r, sigma=sigma, T=T, t=t)

# create df
greeks_df = pd.DataFrame(index=["Delta", "Gamma", "Vega", "Rho", "Theta"])

# fill call column
greeks_df["Call"] = [cp_greeks["delta_call"],cp_greeks["gamma"],cp_greeks["vega"],
                     cp_greeks["rho_call"],cp_greeks["theta_call"]]

# fill put column
greeks_df["Put"] = [cp_greeks["delta_put"],cp_greeks["gamma"],cp_greeks["vega"],
                    cp_greeks["rho_put"],cp_greeks["theta_put"]]

# fill long straddle column
greeks_df["Long Straddle"] = [st_greeks["delta_ls"],st_greeks["gamma_ls"],
                              st_greeks["vega_ls"],st_greeks["rho_ls"],
                              st_greeks["theta_ls"]]

# display df
display(greeks_df)



,Call,Put,Long Straddle
Delta,0.546596,-0.453404,0.093192
Gamma,0.092136,0.092136,0.184272
Vega,11.359217,11.359217,22.718434
Rho,4.334365,-3.851105,0.483261
Theta,-13.002025,-8.022530,-21.024555


At the given parameterization (ATM, one-month maturity, moderate volatility), the call option exhibits a positive delta of 0.547, indicating meaningful upside exposure to the underlying, while the put’s delta of −0.453 reflects symmetric downside sensitivity. The long straddle is close to delta-neutral (0.093), as expected near the money. Both the call and put have identical gamma (0.092) and vega (11.36), reflecting equal convexity and volatility sensitivity under Black–Scholes symmetry, while the straddle doubles these exposures (gamma 0.184, vega 22.72), making it highly sensitive to both price curvature and volatility changes. Rho is positive for the call (4.33) and negative for the put (−3.85), with the straddle net positive (0.48). Theta is negative for both options, with the call at −13.00 and the put at −8.02, and the straddle exhibiting the largest time decay (−21.02), highlighting the cost of holding long volatility positions through time.

# Simulation

Perform Monte Carlo simulation of a stock, assuming the stock follows a GBM process:

$$
dS_t = rS_t\,dt + \sigma S_t\,dB_t^{Q},
\qquad
S_T = S_0 \exp\!\left( (r - 0.5\sigma^2)T + B_T^{Q} \right).
$$

The simulation process is as follows:

1. Draw $N$ Brownian motions $\left\{\left(B_T^{Q}\right)^{(n)}\right\}_{n=1}^{N}$.
2. For each simulated path, the option price is

$$
C^{(n)} = e^{-rT}\max\left(S_T^{(n)} - K, 0\right).
$$

3. Compute the estimate $\frac{1}{N}\sum_{n=1}^{N} C^{(n)}$.

We examine convergence to the Black–Scholes prices as $N$ increases.


In [2]:
# define a function to generate terminal Brownian motion values
def BM(T, t, N=100_000, rng=None):
    """
    Generate an array of N Brownian motions (terminal value B_{T-t}).
    Args:
    -----
          T: float, maturity time
          t: float, current time
          N: int, number of draws (default is 100k)
          rng: numpy random generator (optional)
    Returns:
    --------
          bm: np.ndarray, shape (N,), B_{T-t} ~ Normal(0, T-t)
    """

    # compute time increment
    tau = T - t

    # validate inputs
    if tau <= 0:
        raise ValueError("Require T > t so that (T-t) is positive.")
    if rng is None:
        rng = np.random.default_rng()

    # draw terminal Brownian values
    bm = rng.normal(loc=0.0, scale=np.sqrt(tau), size=N)

    # return draws
    return bm


# define a function to simulate terminal stock values under GBM
def simulate_terminal_stock(S_0, r, sigma, T, N_Sim, t=0, rng=None):
    """
    Simulate terminal stock price S_T under risk-neutral GBM:
        S_T = S_0 * exp((r - 0.5*sigma^2)*(T-t) + sigma * B_{T-t})
    Args:
    -----
          S_0: float, initial spot price
          r: float, interest rate (continuously compounded)
          sigma: float, volatility
          T: float, maturity time
          N_Sim: int, number of Monte Carlo draws
          t: float, current time
          rng: numpy random generator (optional)
    Returns:
    --------
          S_T: np.ndarray, simulated terminal stock values
    """

    # compute time to maturity
    tau = T - t

    # validate inputs
    if tau <= 0:
        raise ValueError("Require T > t so that time-to-maturity (T-t) is positive.")
    if S_0 <= 0:
        raise ValueError("Require S_0 > 0.")
    if sigma <= 0:
        raise ValueError("Require sigma > 0.")

    # draw Brownian terminal values
    B_tau = BM(T=T, t=t, N=N_Sim, rng=rng)

    # compute drift
    drift = (r - 0.5 * sigma**2) * tau

    # compute diffusion
    diffusion = sigma * B_tau

    # compute terminal stock values
    S_T = S_0 * np.exp(drift + diffusion)

    # return
    return S_T


# define a function to compute discounted Monte Carlo call payoffs
def call_monte_carlo(S_0, r, sigma, T, K, N_Sim, t=0, rng=None):
    """
    Perform Monte Carlo simulation for a European call option under risk-neutral GBM.
    Args:
    -----
          S_0: float, initial spot price
          r: float, interest rate (continuously compounded)
          sigma: float, volatility
          T: float, maturity time
          K: float, strike
          N_Sim: int, number of Monte Carlo draws
          t: float, current time
          rng: numpy random generator (optional)
    Returns:
    --------
          C: np.ndarray, discounted call payoffs
    """

    # simulate terminal stock prices
    S_T = simulate_terminal_stock(S_0=S_0, r=r, sigma=sigma, T=T, N_Sim=N_Sim, t=t, rng=rng)

    # compute undiscounted call payoff
    payoff = np.maximum(S_T - K, 0.0)

    # compute discounted payoff
    C = np.exp(-r * (T - t)) * payoff

    # return discounted call payoffs
    return C


# define a function to compute discounted Monte Carlo put payoffs
def put_monte_carlo(S_0, r, sigma, T, K, N_Sim, t=0, rng=None):
    """
    Perform Monte Carlo simulation for a European put option under risk-neutral GBM.
    Args:
    -----
          S_0: float, initial spot price
          r: float, interest rate (continuously compounded)
          sigma: float, volatility
          T: float, maturity time
          K: float, strike
          N_Sim: int, number of Monte Carlo draws
          t: float, current time
          rng: numpy random generator (optional)
    Returns:
    --------
          P: np.ndarray, discounted put payoffs
    """

    # simulate terminal stock prices
    S_T = simulate_terminal_stock(S_0=S_0, r=r, sigma=sigma, T=T, N_Sim=N_Sim, t=t, rng=rng)

    # compute undiscounted put payoff
    payoff = np.maximum(K - S_T, 0.0)

    # discount payoffs
    P = np.exp(-r * (T - t)) * payoff

    # return discounted payoffs
    return P



# simulation count grid
N_sims = [100, 1_000, 10_000, 100_000, 10_000_000]

# create df for call results
monte_carlo_df_call = pd.DataFrame(index=["Mean", "Std. Dev", "Std. Err"])

# create df for put results
monte_carlo_df_put = pd.DataFrame(index=["Mean", "Std. Dev", "Std. Err"])

# set seed for reproducability
rng = np.random.default_rng(123)

# loop through simulation counts
for N_s in N_sims:

    # compute discounted call payoff draws
    call_payoffs = call_monte_carlo(S_0=S, r=r, sigma=sigma, T=T, K=K, N_Sim=N_s, t=t, rng=rng)

    # compute discounted put payoff draws
    put_payoffs = put_monte_carlo(S_0=S, r=r, sigma=sigma, T=T, K=K, N_Sim=N_s, t=t, rng=rng)

    # compute call mean
    call_mean = np.mean(call_payoffs)

    # compute call sample standard deviation
    call_sd = np.std(call_payoffs, ddof=1)

    # compute call standard error
    call_se = call_sd / np.sqrt(N_s)

    # store call stats
    monte_carlo_df_call[f"N={N_s}"] = [call_mean, call_sd, call_se]

    # compute put mean
    put_mean = np.mean(put_payoffs)

    # compute put sample standard deviation
    put_sd = np.std(put_payoffs, ddof=1)

    # compute put standard error
    put_se = put_sd / np.sqrt(N_s)

    # store put stats
    monte_carlo_df_put[f"N={N_s}"] = [put_mean, put_sd, put_se]

# display both tables
print("\n=== Monte Carlo Call Results ===")
display(monte_carlo_df_call)
print("\n=== Monte Carlo Put Results ===")
display(monte_carlo_df_put)


# compute analytic call/put prices
c_0, p_0 = black_scholes(S=S, K=K, r=r, sigma=sigma, T=T, t=t)

# check analytic results
print("\n=== Analytic Call, Put BS Prices ===")
print(c_0, p_0)


=== Monte Carlo Call Results ===


,N=100,N=1000,N=10000,N=100000,N=10000000
Mean,1.936087,1.972376,1.919698,1.918035,1.924641
Std. Dev,2.629173,2.696446,2.708207,2.712512,2.711783
Std. Err,0.262917,0.085269,0.027082,0.008578,0.000858



=== Monte Carlo Put Results ===


,N=100,N=1000,N=10000,N=100000,N=10000000
Mean,1.157990,1.541301,1.505692,1.513581,1.514165
Std. Dev,2.006631,2.403226,2.291467,2.308502,2.306247
Std. Err,0.200663,0.075997,0.022915,0.007300,0.000729



=== Analytic Call, Put BS Prices ===
1.9248158381997058 1.5147002146201416


The Monte Carlo estimates exhibit clean convergence to the analytic Black–Scholes prices as the number of simulations increases. For both the call and the put, the sample mean stabilizes around the closed-form value, with discrepancies at small N driven by sampling noise rather than bias. Although the standard deviation of payoffs remains essentially constant across N, the standard error of the estimator shrinks at the theoretical $\frac{1}{N}$ rate, consistent with the CLT, e.g. the call standard error declines from 0.263 at N=100 to below 0.001 at $N=10^{7}$. By 10MM draws, the Monte Carlo prices are virtually indistinguishable from the analytic Black–Scholes values, illustrating both the unbiasedness and consistency of the Monte Carlo estimator under the risk-neutral GBM model.